In [ ]:
!pip install openai-whisper -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 13.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import numpy as np
import pandas as pd
import torch
import whisper
import librosa
import soundfile as sf
from pathlib import Path
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

from google.colab import drive
drive.mount('/content/drive')

Device: cuda
Mounted at /content/drive


In [ ]:
import os

for root, dirs, files in os.walk("/content/drive"):
    for f in files:
        if f.endswith("_mixed.wav"):
            print(os.path.join(root, f))

/content/drive/MyDrive/ami_audio/EN2001a_mixed.wav
/content/drive/MyDrive/ami_audio/EN2002a_mixed.wav
/content/drive/MyDrive/ami_audio/EN2003a_mixed.wav
/content/drive/MyDrive/ami_audio/EN2004a_mixed.wav
/content/drive/MyDrive/ami_audio/EN2005a_mixed.wav
/content/drive/MyDrive/ami_audio/EN2009b_mixed.wav
/content/drive/MyDrive/ami_audio/IB4001_mixed.wav
/content/drive/MyDrive/ami_audio/IN1001_mixed.wav
/content/drive/MyDrive/ami_audio/IS1000a_mixed.wav
/content/drive/MyDrive/ami_audio/TS3003a_mixed.wav


In [ ]:
# load whisper model
model = whisper.load_model("base", device=device)
print(f"Whisper base loaded on {device}")

# load diarization output
df_diar = pd.read_csv("/content/drive/MyDrive/diarization_output.csv")
print(f"Diarization segments: {len(df_diar)}")
print(df_diar.head())

# paths
audio_dir = Path("/content/drive/MyDrive/ami_audio")
output_dir = Path("/content/drive/MyDrive/transcripts")
output_dir.mkdir(exist_ok=True)

test_meetings  = ['IS1000a', 'TS3003a']
all_meetings   = ['EN2001a','EN2002a','EN2003a','EN2004a','EN2005a',
                  'EN2009b','IB4001','IN1001','IS1000a','TS3003a']

print(f"\nAudio files in Drive:")
for f in sorted(audio_dir.glob("*.wav")):
    print(f"  {f.name}")

100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 111MiB/s]


Whisper base loaded on cuda
Diarization segments: 2033
   meeting  start   end    speaker  duration  split
0  EN2001a    1.5   9.0  Speaker_4       7.5  TRAIN
1  EN2001a    9.0  18.0  Speaker_2       9.0  TRAIN
2  EN2001a   18.0  24.0  Speaker_4       6.0  TRAIN
3  EN2001a   24.0  30.0  Speaker_2       6.0  TRAIN
4  EN2001a   30.0  39.0  Speaker_1       9.0  TRAIN

Audio files in Drive:
  EN2001a_mixed.wav
  EN2002a_mixed.wav
  EN2003a_mixed.wav
  EN2004a_mixed.wav
  EN2005a_mixed.wav
  EN2009b_mixed.wav
  IB4001_mixed.wav
  IN1001_mixed.wav
  IS1000a_mixed.wav
  TS3003a_mixed.wav


In [ ]:
def transcribe_meeting(meeting, model, audio_dir):
    """
    Transcribe full meeting audio with Whisper
    Returns list of word-level segments with timestamps
    """
    audio_path = audio_dir / f"{meeting}_mixed.wav"

    print(f"Transcribing {meeting} ({audio_path.stat().st_size / 1e6:.1f} MB)...")

    result = model.transcribe(
        str(audio_path),
        language="en",
        word_timestamps=True,    # get word-level timestamps
        verbose=False
    )

    # extract word-level data
    words = []
    for segment in result["segments"]:
        if "words" in segment:
            for w in segment["words"]:
                words.append({
                    "word":  w["word"].strip(),
                    "start": w["start"],
                    "end":   w["end"]
                })

    print(f"  Done — {len(words)} words, {len(result['segments'])} segments")
    return words, result

# test on shortest meeting first
test_meeting = "TS3003a"  # 25 min — shortest
words, result = transcribe_meeting(test_meeting, model, audio_dir)

print(f"\nFirst 10 words:")
for w in words[:10]:
    print(f"  {w['start']:6.2f}s - {w['end']:6.2f}s : {w['word']}")

Transcribing TS3003a (48.2 MB)...


100%|██████████| 150564/150564 [00:48<00:00, 3101.43frames/s]

  Done — 2119 words, 346 segments

First 10 words:
   13.28s -  14.00s : Good
   14.00s -  14.72s : morning.
   14.92s -  15.50s : Good
   15.50s -  15.78s : morning.
   16.32s -  16.84s : I
   16.84s -  17.08s : see
   17.08s -  17.26s : you
   17.26s -  17.40s : all
   17.40s -  17.78s : find
   17.78s -  17.98s : your


In [ ]:
def align_words_to_speakers(words, df_diar, meeting):

    meeting_segs = df_diar[df_diar["meeting"] == meeting].reset_index(drop=True)

    aligned = []
    for w in words:
        word_mid = (w["start"] + w["end"]) / 2
        speaker  = "unknown"

        for _, seg in meeting_segs.iterrows():
            if seg["start"] <= word_mid <= seg["end"]:
                speaker = seg["speaker"]
                break

        aligned.append({
            "word":    w["word"],
            "start":   w["start"],
            "end":     w["end"],
            "speaker": speaker
        })

    return aligned

In [ ]:

def group_into_utterances(aligned_words):

    if not aligned_words:
        return []

    utterances = []
    current    = {
        "speaker": aligned_words[0]["speaker"],
        "start":   aligned_words[0]["start"],
        "end":     aligned_words[0]["end"],
        "words":   [aligned_words[0]["word"]]
    }

    for w in aligned_words[1:]:
        if w["speaker"] == current["speaker"]:
            current["end"] = w["end"]
            current["words"].append(w["word"])
        else:
            current["text"] = " ".join(current["words"])
            utterances.append(current)
            current = {
                "speaker": w["speaker"],
                "start":   w["start"],
                "end":     w["end"],
                "words":   [w["word"]]
            }

    current["text"] = " ".join(current["words"])
    utterances.append(current)

    return utterances



In [ ]:
# test on TS3003a
aligned    = align_words_to_speakers(words, df_diar, test_meeting)
utterances = group_into_utterances(aligned)

print(f"Total utterances: {len(utterances)}")
print(f"\nFirst 15 utterances:")
print(f"{'Start':>8} {'End':>8} {'Speaker':>12}  Text")
print("-" * 80)
for u in utterances[:15]:
    text_preview = u["text"][:60] + "..." if len(u["text"]) > 60 else u["text"]
    print(f"{u['start']:>8.1f} {u['end']:>8.1f} {u['speaker']:>12}  {text_preview}")

Total utterances: 143

First 15 utterances:
   Start      End      Speaker  Text
--------------------------------------------------------------------------------
    13.3     18.0    Speaker_0  Good morning. Good morning. I see you all find your
    18.0     23.6    Speaker_2  places. Everybody's sitting on the right place. I guess
    23.6     33.1    Speaker_0  so. Let's see.
    35.4     39.0    Speaker_2  First I'll introduce myself. I don't know if
    39.0     44.0    Speaker_1  everybody knows me. I'm Bart. Hello. Hello. Bart.
    49.8     69.1    Speaker_2  Hello. Let's see. Let's start off with a little presentation...
    69.1     76.8    Speaker_1  few cameras here. They'll record our actions. And you all ha...
    77.8     83.8    Speaker_2  There are also some microphones there. But you don't have to...
    83.8     92.8    Speaker_1  because it will disappear when you don't intend to do it. Th...
    92.8    112.8    Speaker_2  folder. There are some notes in it already. 

In [ ]:
def save_transcript(utterances, meeting, output_dir):
    # save as CSV
    rows = []
    for u in utterances:
        rows.append({
            "meeting": meeting,
            "start":   round(u["start"], 2),
            "end":     round(u["end"], 2),
            "speaker": u["speaker"],
            "text":    u["text"]
        })
    df = pd.DataFrame(rows)
    df.to_csv(output_dir / f"{meeting}_transcript.csv", index=False)

    # save as readable TXT
    with open(output_dir / f"{meeting}_transcript.txt", "w") as f:
        f.write(f"Meeting: {meeting}\n")
        f.write("=" * 60 + "\n\n")
        for u in utterances:
            f.write(f"[{u['start']:.1f}s - {u['end']:.1f}s] {u['speaker']}\n")
            f.write(f"{u['text']}\n\n")

    print(f"  Saved {meeting}_transcript.csv and .txt")
    return df


# save TS3003a first
save_transcript(utterances, test_meeting, output_dir)

# now run all meetings
all_transcripts = []

for meeting in tqdm(all_meetings, desc="Transcribing meetings"):
    audio_path = audio_dir / f"{meeting}_mixed.wav"
    if not audio_path.exists():
        print(f"  {meeting} — audio not found, skipping")
        continue

    try:
        words, result  = transcribe_meeting(meeting, model, audio_dir)
        aligned        = align_words_to_speakers(words, df_diar, meeting)
        utterances     = group_into_utterances(aligned)
        df_t           = save_transcript(utterances, meeting, output_dir)
        all_transcripts.append(df_t)
        print(f"  {meeting}: {len(utterances)} utterances")
    except Exception as e:
        print(f"  {meeting} failed: {e}")


  Saved TS3003a_transcript.csv and .txt


Transcribing meetings:   0%|          | 0/10 [00:00<?, ?it/s]

Transcribing EN2001a (168.0 MB)...



100%|██████████| 525024/525024 [03:29<00:00, 2500.25frames/s]


  Done — 13504 words, 957 segments


...

  Saved IS1000a_transcript.csv and .txt
  IS1000a: 148 utterances
Transcribing TS3003a (48.2 MB)...



100%|██████████| 150564/150564 [00:40<00:00, 3747.75frames/s]


  Done — 2103 words, 333 segments


Transcribing meetings: 100%|██████████| 10/10 [27:01<00:00, 162.18s/it]

  Saved TS3003a_transcript.csv and .txt
  TS3003a: 143 utterances


In [ ]:

# combine all into one CSV
df_all = pd.concat(all_transcripts, ignore_index=True)
df_all.to_csv("/content/drive/MyDrive/transcripts/all_transcripts.csv", index=False)

print(f"\nTotal utterances across all meetings: {len(df_all)}")
print(df_all.head(10))


Total utterances across all meetings: 2101
   meeting  start    end    speaker  \
0  EN2001a   2.98   5.94  Speaker_4   
1  EN2001a  11.12  17.72  Speaker_2   
2  EN2001a  17.72  24.10  Speaker_4   
3  EN2001a  24.10  29.66  Speaker_2   
4  EN2001a  29.66  38.92  Speaker_1   
5  EN2001a  41.90  48.30  Speaker_2   
6  EN2001a  48.30  53.42  Speaker_4   
7  EN2001a  53.42  62.90  Speaker_2   
8  EN2001a  62.90  68.96  Speaker_4   
9  EN2001a  68.96  74.98  Speaker_2   

                                                text  
0                                        Okay. Okay.  
1  Does anyone want to see Steve's feedback from ...  
2  than he said yesterday? Not really. Just what ...  
3  duplication of effort. But duplication of effo...  
4  and it's a shame that we should maybe thank yo...  
5          So, it's probably pilot times. I'd say if  
6  for the prototype we see just like wherever po...  
7  pre annotated and stuff and for the stuff that...  
8  on the interface first sort 